In [ ]:
import os
import warnings
# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import cv2
import mediapipe as mp
import time
from directkeys import right_pressed, left_pressed
from directkeys import PressKey, ReleaseKey

break_key_pressed = left_pressed
accelerato_key_pressed = right_pressed

print("Starting in 3 seconds...")
time.sleep(3.0)
current_key_pressed = set()

mp_draw = mp.solutions.drawing_utils
mp_hand = mp.solutions.hands

tipIds = [4, 8, 12, 16, 20]

# Initialize video capture
video = cv2.VideoCapture(0)

# Check if camera is working
if not video.isOpened():
    print("Error: Could not open camera")
    exit()

# Set camera properties for better performance
video.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
video.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
video.set(cv2.CAP_PROP_FPS, 30)

print("Camera initialized. Press 'q' to quit.")

with mp_hand.Hands(
    min_detection_confidence=0.7,  # Increased for better detection
    min_tracking_confidence=0.7,   # Increased for better tracking
    max_num_hands=1,
    static_image_mode=False,
    model_complexity=1
) as hands:
    
    frame_count = 0
    while True:
        keyPressed = False
        break_pressed = False
        accelerator_pressed = False
        key_count = 0
        key_pressed = 0
        
        ret, image = video.read()
        if not ret:
            print("Failed to read frame")
            continue
            
        frame_count += 1
        
        # Flip the image horizontally for a mirror effect
        image = cv2.flip(image, 1)
        
        # Convert BGR to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_rgb.flags.writeable = False
        
        # Process the image
        results = hands.process(image_rgb)
        
        # Convert back to BGR
        image_rgb.flags.writeable = True
        image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
        
        # Draw hand landmarks
        lmList = []
        if results.multi_hand_landmarks:
            for hand_landmark in results.multi_hand_landmarks:
                # Get landmarks
                for id, lm in enumerate(hand_landmark.landmark):
                    h, w, c = image.shape
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    lmList.append([id, cx, cy])
                
                # Draw landmarks
                mp_draw.draw_landmarks(image, hand_landmark, mp_hand.HAND_CONNECTIONS)
        
        # Count fingers
        fingers = []
        if len(lmList) != 0:
            # Thumb (check x coordinate because image is flipped)
            if lmList[tipIds[0]][1] < lmList[tipIds[0] - 1][1]:  # Changed > to < because of flip
                fingers.append(1)
            else:
                fingers.append(0)
            
            # Other four fingers (check y coordinate)
            for id in range(1, 5):
                if lmList[tipIds[id]][2] < lmList[tipIds[id] - 2][2]:
                    fingers.append(1)
                else:
                    fingers.append(0)
            
            total = fingers.count(1)
            
            # Display finger count for debugging
            cv2.putText(image, f'Fingers: {total}', (10, 50), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            
            # BRAKE (0 fingers - closed fist)
            if total == 0:
                cv2.rectangle(image, (20, 300), (270, 425), (0, 255, 0), cv2.FILLED)
                cv2.putText(image, "BRAKE", (45, 375), cv2.FONT_HERSHEY_SIMPLEX,
                          2, (255, 0, 0), 5)
                
                if break_key_pressed not in current_key_pressed:
                    PressKey(break_key_pressed)
                    current_key_pressed.add(break_key_pressed)
                
                keyPressed = True
                print("BRAKE activated")
                
            # GAS (5 fingers - open hand)
            elif total == 5:
                cv2.rectangle(image, (20, 300), (270, 425), (0, 255, 0), cv2.FILLED)
                cv2.putText(image, "GAS", (45, 375), cv2.FONT_HERSHEY_SIMPLEX,
                          2, (255, 0, 0), 5)
                
                if accelerato_key_pressed not in current_key_pressed:
                    PressKey(accelerato_key_pressed)
                    current_key_pressed.add(accelerato_key_pressed)
                
                keyPressed = True
                print("GAS activated")
            
            # NEUTRAL (any other finger count)
            else:
                cv2.rectangle(image, (20, 300), (270, 425), (255, 255, 0), cv2.FILLED)
                cv2.putText(image, "NEUTRAL", (25, 375), cv2.FONT_HERSHEY_SIMPLEX,
                          1.5, (0, 0, 255), 5)
        
        else:
            # No hand detected
            cv2.putText(image, 'No hand detected', (10, 100), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
        # Release keys if no gesture is detected
        if not keyPressed and len(current_key_pressed) != 0:
            for key in current_key_pressed:
                ReleaseKey(key)
            current_key_pressed.clear()
            print("Keys released")
        
        # Add frame counter for debugging
        cv2.putText(image, f'Frame: {frame_count}', (10, image.shape[0] - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Show the frame
        cv2.imshow("Hand Gesture Control", image)
        
        # Break on 'q' key press
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            print("Exiting...")
            break

# Clean up
for key in current_key_pressed:
    ReleaseKey(key)

video.release()
cv2.destroyAllWindows()
print("Camera released and windows closed.")


Starting in 3 seconds...
Camera initialized. Press 'q' to quit.
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
Keys released
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
Keys released
GAS activated
Keys released
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
GAS activated
Keys released
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE activated
BRAKE 